# 第 4 节：马尔可夫决策过程（MDP）

---

## 📍 本节在知识体系中的位置

```
RL 概述 (01) → 数学准备 (02) → 多臂老虎机 (03) → **MDP (04)** → Bellman 方程 (05) → MC (06) → TD (07) → ...
                                                    ↑
                                              你在这里
```

MDP 是强化学习的**数学语言**。几乎所有 RL 问题都可以形式化为 MDP。
理解 MDP 是理解后续所有算法的基础。

---

## 🎯 学习目标

完成本节后，你将能够：

1. 解释 Markov 性质的含义及其在 RL 中的作用
2. 写出 MDP 的五元组定义
3. 理解状态转移概率、奖励函数和折扣因子
4. 从给定 MDP 生成轨迹
5. 计算轨迹的回报
6. 理解策略的数学定义及其分类
7. 区分 model-based 和 model-free 方法
8. 理解 episodic 和 continuing task 的区别


## 1. Markov 性质（马尔可夫性）

### 直觉

> **"未来只取决于现在，不取决于过去。"**

在 RL 中，这意味着：给定当前状态，过去的历史对于预测未来是**无关**的。

### 数学定义

一个状态 $s_t$ 具有 Markov 性质，当且仅当：

$$P(s_{t+1} | s_t, a_t) = P(s_{t+1} | s_0, a_0, s_1, a_1, \ldots, s_t, a_t)$$

即：下一状态 $s_{t+1}$ 的条件概率分布**仅依赖于**当前状态 $s_t$ 和当前动作 $a_t$，
与之前的历史轨迹无关。

### 为什么 Markov 性质重要？

1. **数学可处理性**：如果状态是 Markov 的，我们可以用有限维的转移矩阵完全描述环境动态
2. **计算可行性**：Bellman 方程（下节）依赖 Markov 性质
3. **状态设计指南**：在实践中，我们需要设计包含足够信息的**状态表示**

### 例子

- ✅ **Markov**：棋类游戏（棋盘状态包含所有信息）
- ✅ **Markov**：物理系统（位置+速度决定未来）
- ⚠️ **可能非 Markov**：单一当前股价通常不足以作为 Markov 状态（受不可观测因素影响）；更丰富的状态表示（如多时间尺度的量价特征）可能近似 Markov 性质。一个过程是否 Markov 取决于选取的状态表示是否包含预测未来所需的信息。
- ⚠️ **近似 Markov**：Atari 游戏中用连续 4 帧作为状态（因为单帧不知道速度）


## 2. MDP 的形式化定义

一个**有限 MDP** 由五元组 $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$ 定义：

### 2.1 状态空间 $\mathcal{S}$

所有可能状态的集合。在有限 MDP 中，$|\mathcal{S}| = n$。

- 例：GridWorld 中每个格子是一个状态，$\mathcal{S} = \{0, 1, \ldots, 15\}$

### 2.2 动作空间 $\mathcal{A}$

所有可能动作的集合。$|\mathcal{A}| = m$。

- 例：GridWorld 中 $\mathcal{A} = \{0(上), 1(右), 2(下), 3(左)\}$

### 2.3 状态转移概率 $P$

$$P(s' | s, a) = \Pr\{S_{t+1} = s' \mid S_t = s, A_t = a\}$$

即：在状态 $s$ 执行动作 $a$ 后，转移到状态 $s'$ 的概率。

**性质**：
- $0 \leq P(s' | s, a) \leq 1$（概率约束）
- $\sum_{s'} P(s' | s, a) = 1$（归一化条件）

转移概率 $P$ 定义了一个 **$|\mathcal{S}| \times |\mathcal{A}| \times |\mathcal{S}|$** 的张量。

### 2.4 奖励函数 $R$

$$R(s, a) = \mathbb{E}[R_{t+1} \mid S_t = s, A_t = a]$$

即：在状态 $s$ 执行动作 $a$ 后获得的**期望**即时奖励。

奖励函数的常见形式：
- $R(s, a)$：状态-动作对 → 标量
- $R(s, a, s')$：包含下一状态
- $r(s)$：仅依赖状态

### 2.5 折扣因子 $\gamma$

$$\gamma \in [0, 1]$$

折扣因子决定**未来奖励在当前的价值**。

- $\gamma = 0$：只关心即时奖励（短视）
- $\gamma = 1$：所有时间步的奖励同等重要（远视）
- $\gamma \to 1$：典型值如 0.99

**$\gamma < 1$ 的三个原因**：
1. 数学方便：保证无限序列回报收敛
2. 不确定性：未来不完全可预测
3. 实际偏好：人类和动物更偏好即时奖励


## 3. 轨迹与回报

### 3.1 轨迹（Trajectory / Episode）

一条轨迹 $\tau$ 是智能体与环境交互的序列：

$$\tau = (s_0, a_0, r_1, s_1, a_1, r_2, s_2, \ldots, s_{T-1}, a_{T-1}, r_T, s_T)$$

其中 $T$ 是轨迹长度（可能为 $\infty$）。

### 3.2 回报（Return）

回报 $G_t$ 是从时间步 $t$ 开始的**累积折扣奖励**：

$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

**递推关系**（本节最重要的恒等式之一）：

$$G_t = R_{t+1} + \gamma G_{t+1}$$

这个递推关系是后续 Bellman 方程推导的关键。

### 3.3 Episodic vs Continuing Tasks

| 类型 | 特点 | 例子 | 处理 |
|------|------|------|------|
| **Episodic** | 有终止状态，有限步数 | 游戏通关、机器人到达目标 | 自然终止 |
| **Continuing** | 无终止状态，无限持续 | 机器人导航、股票交易 | 常使用 $\gamma < 1$ 保证收敛；也可使用 average-reward 形式 |


## 4. 策略（Policy）

### 4.1 定义

策略 $\pi$ 是智能体的**行为规则**——在每个状态下选择动作的方式。

**随机策略**：

$$\pi(a | s) = \Pr\{A_t = a \mid S_t = s\}$$

- $\pi: \mathcal{S} \times \mathcal{A} \to [0, 1]$
- $\sum_{a \in \mathcal{A}} \pi(a|s) = 1$（概率归一化）

**确定性策略**：

$$a = \pi(s)$$

- $\pi: \mathcal{S} \to \mathcal{A}$

确定性策略是随机策略的特例（某个动作概率为 1，其余为 0）。

### 4.2 为什么需要随机策略？

1. **探索**：随机性允许探索未知的状态-动作对
2. **部分可观测环境 (POMDP)**：当状态不完全可观测时，随机策略可能优于确定性策略
3. **探索需要**：训练阶段需要随机性来探索未知的状态-动作对

> 注意：在标准有限 MDP（完全可观测、有限状态/动作、折扣回报）中，至少存在一个确定性平稳最优策略。随机策略的需求通常来自部分可观测、博弈论场景或多智能体设定。


## 5. 代码实践：策略与轨迹

让我们用代码展示策略、MDP 和轨迹的关系。


In [ ]:
import numpy as np
import sys
sys.path.insert(0, '/workspace/data/vggt-omega/rl')

from rl_course.envs.grid_world import GridWorld
from rl_course.utils.seeding import set_seed

set_seed(42)

# 创建一个 4×4 的 GridWorld
gw = GridWorld(width=4, height=4, step_reward=-1.0, goal_reward=10.0)
print(f"状态数: {gw.n_states}, 动作数: {gw.n_actions}")
print("环境:")
print(gw.render(mode='ansi'))


### 5.1 随机策略生成轨迹

In [ ]:
def random_policy(state: int, n_actions: int = 4) -> int:
    '''随机策略：每个动作等概率'''
    return np.random.randint(n_actions)

def generate_trajectory(env, policy_fn, max_steps=100):
    '''生成一条轨迹

    Returns:
        states, actions, rewards: 轨迹的三个列表
        total_return: 折扣回报 G_0
    '''
    states, actions, rewards = [], [], []
    state = env.reset()
    done = False

    while not done and len(states) < max_steps:
        action = policy_fn(state)
        next_state, reward, done, _ = env.step(action)

        states.append(state)
        actions.append(action)
        rewards.append(reward)

        state = next_state

    # 计算回报（从 t=0 开始）
    gamma = 0.99
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G

    return states, actions, rewards, G

# 生成 3 条轨迹
for i in range(3):
    states, actions, rewards, G = generate_trajectory(gw, random_policy)
    print(f"轨迹 {i+1}: {len(states)} 步, 回报 G={G:.2f}, 奖励序列={rewards}")


### 5.2 转移矩阵可视化

每个 MDP 都有一个转移矩阵 $P$，shape 为 $(|\mathcal{S}|, |\mathcal{A}|, |\mathcal{S}|)$。

$P[s, a, s']$ = 在状态 $s$ 执行动作 $a$ 后转移到 $s'$ 的概率。


In [ ]:
# 获取确定性转移矩阵
P = gw.get_transition_matrix()
print(f"转移矩阵 shape: {P.shape}  (n_states={P.shape[0]}, n_actions={P.shape[1]}, n_next_states={P.shape[2]})")

# 展示状态 0（左上角）的转移
print("\n状态 0 (左上角) 的转移:")
for a in range(gw.n_actions):
    probs = P[0, a]
    next_states = np.where(probs > 0)[0]
    for ns in next_states:
        print(f"  动作 {a} ({gw.ACTION_NAMES[a]}): → 状态 {ns} (概率 {probs[ns]:.2f})")

# 验证每行的概率和为 1
for s in range(gw.n_states):
    for a in range(gw.n_actions):
        assert abs(P[s, a].sum() - 1.0) < 1e-6, f"概率和 != 1 at state {s}, action {a}"

print("\n✅ 所有转移概率和 = 1")


### 5.3 奖励矩阵

In [ ]:
# 获取奖励矩阵
R = gw.get_reward_matrix()
print(f"奖励矩阵 shape: {R.shape}")

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 转移矩阵 - 展示某个状态的转移
s_demo = 0
P_s = P[s_demo]  # shape (4, 16)
im1 = axes[0].imshow(P_s, aspect='auto', cmap='Blues')
axes[0].set_title(f'转移矩阵 P[{s_demo}, :, :]\\n(状态 {s_demo} 的各动作转移)')
axes[0].set_xlabel('下一状态 s\'')
axes[0].set_ylabel('动作 a')
axes[0].set_xticks(range(0, 16, 4))
axes[0].set_yticks(range(4))
axes[0].set_yticklabels(gw.ACTION_NAMES)
plt.colorbar(im1, ax=axes[0])

# 奖励矩阵热力图
im2 = axes[1].imshow(R.T, aspect='auto', cmap='RdYlGn')
axes[1].set_title('奖励矩阵 R (期望奖励)')
axes[1].set_xlabel('状态 s')
axes[1].set_ylabel('动作 a')
axes[1].set_yticks(range(4))
axes[1].set_yticklabels(gw.ACTION_NAMES)
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.savefig('outputs/figures/04_mdp_transition_reward.png', dpi=100)
plt.close()
print("✅ 图已保存到 outputs/figures/04_mdp_transition_reward.png")


### 5.4 随机转移（Slippery GridWorld）

现实中的 MDP 往往具有随机转移。在 RL 中，我们对**期望**建模：

$$R(s, a) = \mathbb{E}[R_{t+1} | S_t = s, A_t = a]$$
$$P(s' | s, a) = \Pr\{S_{t+1} = s' | S_t = s, A_t = a\}$$

也就是说，MDP 只需要知道**概率分布**，而不需要知道具体的随机实现。


In [ ]:
from rl_course.envs.grid_world import StochasticGridWorld

# 创建随机 GridWorld：30% 概率执行随机动作
sgw = StochasticGridWorld(width=4, height=4, slip_prob=0.3, seed=42)
P_stoch = sgw.get_transition_matrix()

# 对比确定性和随机转移
print("确定性 GridWorld vs 随机 GridWorld (slip=0.3):")
print(f"\n状态 0, 动作 1 (→) 的转移分布:")
for ns in range(sgw.n_states):
    det_p = P[0, 1, ns]  # 确定性转移
    stoch_p = P_stoch[0, 1, ns]  # 随机转移
    if det_p > 0 or stoch_p > 0.01:
        print(f"  → 状态 {ns}: 确定性={det_p:.3f}, 随机={stoch_p:.3f}")

print("\n注意：在随机版本中，即使选择'向右'，也可能滑到其他方向！")


## 6. 折扣因子的影响

不同 $\gamma$ 值对回报的影响：

In [ ]:
def compute_return(rewards, gamma):
    '''计算折扣回报 G_0 = Σ γ^t * r_{t+1}'''
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
    return G

# 奖励序列: 每步 -1，第 10 步 +100
rewards = [-1] * 9 + [100]

gammas = [0.0, 0.5, 0.9, 0.99, 1.0]
print("不同 γ 下的回报:")
print(f"{'γ':<8} {'G_0':<10}")
print("-" * 20)
for g in gammas:
    G = compute_return(rewards, g)
    print(f"{g:<8.2f} {G:<10.4f}")

# 可视化
fig, ax = plt.subplots(figsize=(10, 4))
gamma_range = np.linspace(0, 1, 100)
returns = [compute_return(rewards, g) for g in gamma_range]
ax.plot(gamma_range, returns, linewidth=2)
ax.set_xlabel('γ (折扣因子)')
ax.set_ylabel('G₀ (回报)')
ax.set_title('折扣因子对回报的影响')
ax.axvline(x=0.99, color='r', linestyle='--', alpha=0.5, label='γ=0.99')
ax.grid(True, alpha=0.3)
ax.legend()
plt.savefig('outputs/figures/04_discount_factor.png', dpi=100)
plt.close()
print("✅ 图已保存")


## 7. Model-Based vs Model-Free

这是 RL 中最基本的分类之一：

### Model-Based（基于模型）

智能体**知道**或**学习**环境的动态模型（$P$ 和 $R$），然后用这个模型**规划**。

- 知道 $P(s'|s,a)$ 和 $R(s,a)$
- 可以做"想象"：在脑中模拟动作的结果
- 代表方法：Dynamic Programming、AlphaZero
- 优点：样本效率高（因为可以利用模型模拟）
- 缺点：需要模型，而模型往往是未知或有误差的

### Model-Free（无模型）

智能体**不建模**环境，直接从经验中学习。

- 不知道 $P$ 和 $R$
- 只能通过实际交互获取经验
- 代表方法：Q-Learning、REINFORCE、PPO
- 优点：不需要建模，更通用
- 缺点：样本效率低（需要大量交互）

### 直观理解

```
Model-Based:  就像下棋时在脑中推演几步
Model-Free:   就像骑自行车（靠感觉，不靠物理方程）
```

本课程主要关注 **Model-Free** 方法，但在本节和第 5 节也会详细讲解 Model-Based 的 Dynamic Programming。


## 8. On-Policy vs Off-Policy

另一个核心分类：

| 特性 | On-Policy | Off-Policy |
|------|-----------|------------|
| **定义** | 学习策略 = 行为策略 | 学习策略 ≠ 行为策略 |
| **学习对象** | 评估和改进同一个策略 | 从任意策略产生的数据中学习最优策略 |
| **代表算法** | SARSA, A2C, PPO, TRPO | Q-Learning, DQN |
| **优点** | 更稳定 | 样本效率更高（可复用旧经验） |
| **缺点** | 样本效率低 | 可能不稳定（重要性采样方差） |

### 例子

- **On-Policy (SARSA)**：你按照自己的策略走，边走边改进这个策略
- **Off-Policy (Q-Learning)**：你看着别人走（或随机走），但学习的是最优走法


## 9. 本节总结

### 核心概念

| 概念 | 符号 | 含义 |
|------|------|------|
| 状态 | $s \in \mathcal{S}$ | 环境的当前情况 |
| 动作 | $a \in \mathcal{A}$ | 智能体的选择 |
| 转移概率 | $P(s'\vert s,a)$ | 环境动态 |
| 奖励 | $R(s,a)$ | 即时反馈 |
| 折扣因子 | $\gamma$ | 未来奖励的权重 |
| 策略 | $\pi(a\vert s)$ | 行为规则 |
| 回报 | $G_t = \sum_k \gamma^k R_{t+k+1}$ | 累积折扣奖励 |
| 轨迹 | $\tau = (s_0,a_0,r_1,s_1,\dots)$ | 交互序列 |

### 关键恒等式

**回报的递推关系**：$G_t = R_{t+1} + \gamma G_{t+1}$

这是下节 Bellman 方程推导的出发点。

### 下一步

下一节我们将学习 **Bellman 方程**，它利用回报的递推关系，建立状态价值的自洽方程。
这是 RL 中最核心的数学工具。


## 10. 面试问题

<details>
<summary><b>Q1: 什么是 Markov 性质？为什么 RL 依赖它？</b></summary>

**30秒回答**：Markov 性质指"未来只取决于现在，不取决于过去"。在 RL 中，这意味着 $P(s_{t+1}|s_t,a_t)$ 完全描述环境动态，使得我们可以用 Bellman 方程递推求解。

**2分钟回答**：Markov 性质是 RL 理论的基础。它允许我们将状态转移建模为 $P(s'|s,a)$，而不需要考虑完整历史。这使得：
1. 状态空间保持有限维度
2. Bellman 方程成立（$V(s) = \max_a [R(s,a) + \gamma \sum_{s'} P(s'|s,a) V(s')]$）
3. 策略可以仅依赖当前状态

在实际中，我们通过构造**包含足够信息的状态表示**来满足（或近似）Markov 性质，例如 Atari DQN 使用连续 4 帧。

**常见追问**：如果环境不是 Markov 的怎么办？
→ 方法：使用 RNN/LSTM 编码历史，或设计包含历史信息的状态特征。
</details>

<details>
<summary><b>Q2: 解释 MDP 五元组和每个元素的含义</b></summary>

$(\mathcal{S}, \mathcal{A}, P, R, \gamma)$：
- $\mathcal{S}$：状态空间（所有可能状态的集合）
- $\mathcal{A}$：动作空间
- $P(s'|s,a)$：状态转移概率（环境动态）
- $R(s,a)$：期望即时奖励函数
- $\gamma \in [0,1]$：折扣因子（权衡短期 vs 长期奖励）
</details>

<details>
<summary><b>Q3: 折扣因子 γ 为什么通常小于 1？</b></summary>

1. **数学收敛**：保证无限序列的回报 $\sum_{t=0}^{\infty} \gamma^t r_t$ 有界
2. **不确定性**：未来不可完全预测，折扣反映了这种不确定性
3. **实际偏好**：人类和动物表现出时间折扣偏好

但 $\gamma=1$ 在 episodic 任务中是可行的（因为有终止状态）。
</details>


## 11. 练习

### 概念题

1. 判断以下环境是否具有 Markov 性质：
   - (a) 国际象棋
   - (b) 只用当前价格预测股价
   - (c) 自动驾驶中使用摄像头+雷达感知

2. 在 GridWorld 中，如果 slip_prob=0.5，这意味着什么？对最优策略有何影响？

### 编程题

3. 在 GridWorld 中添加障碍物，计算新的转移矩阵，观察哪些状态-动作对的转移发生了改变。

4. 实现一个非均匀随机策略（例如偏好向右走），计算在此策略下的轨迹平均回报。


---
*下一节：[05_bellman_dynamic_programming.ipynb](05_bellman_dynamic_programming.ipynb) — Bellman 方程与动态规划*
